In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [ ]:
import sys
import torch
import time
import numpy as np
import torch.nn as nn

from tqdm import tqdm
from collections import defaultdict
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# Assumes row multiples of block size
def quantize_dequantize_mx_vec(tensor, mantissa_bits, block_size):
    max_quant_val = (2**mantissa_bits) - 1
    original_shape = tensor.shape
    # Reshape into independent blocks
    reshaped_tensor = tensor.reshape(-1, block_size)
    max_abs_vals = torch.max(torch.abs(reshaped_tensor), dim=1, keepdim=True).values
    sf = torch.where(max_abs_vals > 0, max_abs_vals / max_quant_val, 1.0)
    quantized_blocks = torch.round(reshaped_tensor / sf)
    dequantized_blocks = quantized_blocks * sf
    return dequantized_blocks.reshape(original_shape)

def add_gaussian_noise(tensor, mean=0.0, sigma=0.01):
    if sigma <= 0:
        return tensor
    noise = torch.randn(tensor.size(), device=tensor.device) * sigma + mean
    return tensor + noise

class SimulatedFFN(nn.Module):

    def __init__(self, original_mlp, tile_shape=(32, 40), noise_mean=0.0, noise_sigma=0.01):
        super().__init__()
        # Extract weights from the original FFN layer
        self.gate_proj = original_mlp.gate_proj
        self.up_proj = original_mlp.up_proj
        self.down_proj = original_mlp.down_proj

        # Simulation parameters
        self.tile_shape = tile_shape
        self.noise_mean = noise_mean
        self.noise_sigma = noise_sigma

    def tiled_mat_mul_vec(self, input_tensor, weight_tensor):
        m, k = input_tensor.shape
        _, n = weight_tensor.T.shape
        tile_m, block_size = self.tile_shape
        tile_n = tile_m  # Use square tiles for output

        #Pad tensors to be divisible by tile dimensions
        pad_m = (tile_m - m % tile_m) % tile_m
        pad_k = (block_size - k % block_size) % block_size
        pad_n = (tile_n - n % tile_n) % tile_n

        inp_pad = nn.functional.pad(input_tensor, (0, pad_k, 0, pad_m))
        w_pad = nn.functional.pad(weight_tensor.T, (0, pad_n, 0, pad_k))
        m_pad, k_pad = inp_pad.shape
        _, n_pad = w_pad.shape

        inp_tiles = inp_pad.unfold(0, tile_m, tile_m).unfold(1, block_size, block_size)
        num_m_tiles, num_k_tiles, _, _ = inp_tiles.shape

        w_tiles = w_pad.unfold(0, block_size, block_size).unfold(1, tile_n, tile_n)
        _, num_n_tiles, _, _ = w_tiles.shape

        inp_dequant = quantize_dequantize_mx_vec(
            inp_tiles.contiguous().view(-1, block_size), mantissa_bits=7, block_size=block_size
        ).view(inp_tiles.shape)

        w_tiles_T = w_tiles.permute(0, 1, 3, 2).contiguous()
        w_dequant_T = quantize_dequantize_mx_vec(
            w_tiles_T.view(-1, block_size), mantissa_bits=2, block_size=block_size
        ).view(w_tiles_T.shape)
        w_dequant = w_dequant_T.permute(0, 1, 3, 2).contiguous()

        partial_products = torch.matmul(
            inp_dequant.unsqueeze(2).to(torch.bfloat16),
            w_dequant.unsqueeze(0).to(torch.bfloat16)
        )

        noisy_partials = add_gaussian_noise(partial_products, self.noise_mean, self.noise_sigma)
        output_tiles = torch.sum(noisy_partials, dim=1).to(input_tensor.dtype)
        out_pad = output_tiles.permute(0, 2, 1, 3).contiguous().view(m_pad, n_pad)
        return out_pad[:m, :n]

    def forward(self, x):
        batch_size, seq_len, hidden_dim = x.shape
        x_2d = x.view(-1, hidden_dim)

        # Gating is done in high precision
        gated_states = self.gate_proj(x_2d)
        gated_states = nn.functional.gelu(gated_states)

        # Up Projection uses simulated matrix multiplication
        up_states = self.tiled_mat_mul_vec(x_2d, self.up_proj.weight)
        activated_states = gated_states * up_states

        # Down Projection uses simulated matrix multiplication
        output_states_2d = self.tiled_mat_mul_vec(activated_states, self.down_proj.weight)
        return output_states_2d.view(batch_size, seq_len, -1)

def replace_ffn_with_simulation(model, noise_mean, noise_sigma):
    for layer in model.model.layers:
        original_mlp = layer.mlp
        layer.mlp = SimulatedFFN(original_mlp, noise_mean=noise_mean, noise_sigma=noise_sigma)
    print("Replacement complete.")
    return model

def format_mmlu_prompt(dev_samples, test_sample, subject_name, k_shot):
    prompt = f"The following are multiple choice questions (with answers) about {subject_name}.\n\n"

    # Add few-shot examples if k_shot > 0
    if k_shot > 0:
        for i in range(k_shot):
            question = dev_samples[i]['question']
            choices_str = "\n".join([f"{chr(65+j)}. {dev_samples[i]['choices'][j]}" for j in range(len(dev_samples[i]['choices']))])
            prompt += f"Question: {question}\n{choices_str}\nAnswer: {chr(65+dev_samples[i]['answer'])}\n\n"

    # Add the actual test question
    question = test_sample['question']
    choices_str = "\n".join([f"{chr(65+j)}. {test_sample['choices'][j]}" for j in range(len(test_sample['choices']))])
    prompt += f"Question: {question}\n{choices_str}\nAnswer:"
    return prompt, chr(65 + test_sample['answer'])

@torch.no_grad()
def evaluate_mmlu(model, tokenizer, device, k_shot=5):
    print(f"Loading MMLU dataset for {k_shot}-shot evaluation...")
    mmlu_dataset = load_dataset("cais/mmlu", "all")
    dev_examples_by_subject = defaultdict(list)
    if k_shot > 0:
        print("Grouping Few-Shot Examples by subject")
        for example in tqdm(mmlu_dataset["dev"], desc="Processing Dev Split"):
            dev_examples_by_subject[example["subject"]].append(example)

    subjects = sorted(list(set(mmlu_dataset["test"]["subject"])))
    results = {}
    total_correct = 0
    total_all = 0

    print("Starting MMLU evaluation...")
    for subject in tqdm(subjects, desc="Evaluating Subjects"):
        print(f"Running for Subject {subject}")
        subject_correct = 0
        subject_test_set = mmlu_dataset["test"].filter(lambda example: example["subject"] == subject)
        subject_dev_samples = dev_examples_by_subject[subject] if k_shot > 0 else None

        for test_sample in subject_test_set:
            # Prepare sample for the prompt
            prompt, correct_answer = format_mmlu_prompt(subject_dev_samples, test_sample, subject.replace("_", " "), k_shot=k_shot)
            inputs = tokenizer(prompt, return_tensors="pt").to(device)

            outputs = model(**inputs)
            logits = outputs.logits[0, -1]
            choice_tokens = [tokenizer.encode(f" {chr(65+j)}", add_special_tokens=False)[-1] for j in range(len(test_sample["choices"]))]
            choice_logits = logits[choice_tokens]
            prediction_idx = torch.argmax(choice_logits).item()
            prediction = chr(65 + prediction_idx)

            if prediction == correct_answer:
                subject_correct += 1
        accuracy = subject_correct / len(subject_test_set)
        results[subject] = accuracy
        print(f"  Subject: {subject}, Accuracy: {accuracy:.4f}")
        total_correct += subject_correct
        total_all += len(subject_test_set)

    overall_accuracy = total_correct / total_all
    print(f"\nOverall MMLU Accuracy ({k_shot}-shot): {overall_accuracy:.4f}")
    return results

In [ ]:

MODEL_ID = "google/gemma-2-9b"

NOISE_MEAN = 0.0
NOISE_SIGMA = 0.0

K_SHOT = 5

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

print("Loading base model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map=device,
)

model = replace_ffn_with_simulation(model, noise_mean=NOISE_MEAN, noise_sigma=NOISE_SIGMA)
evaluate_mmlu(model, tokenizer, device, k_shot=K_SHOT)